Problem Statement: Employee attrition increases hiring costs and reduces productivity. By predicting employees who are likely to leave, organizations can take proactive measures to improve retention and workforce stability.

Task 1 - Data Loading & Exploration 

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, roc_curve)

In [4]:
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')

In [5]:
df.head(10)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2
5,32,No,Travel_Frequently,1005,Research & Development,2,2,Life Sciences,1,8,...,3,80,0,8,2,2,7,7,3,6
6,59,No,Travel_Rarely,1324,Research & Development,3,3,Medical,1,10,...,1,80,3,12,3,2,1,0,0,0
7,30,No,Travel_Rarely,1358,Research & Development,24,1,Life Sciences,1,11,...,2,80,1,1,2,3,1,0,0,0
8,38,No,Travel_Frequently,216,Research & Development,23,3,Life Sciences,1,12,...,2,80,0,10,2,3,9,7,1,8
9,36,No,Travel_Rarely,1299,Research & Development,27,3,Medical,1,13,...,2,80,2,17,3,2,7,7,7,7


In [6]:
row, cols = df.shape

In [7]:
print(row)
print(cols)

1470
35


In [8]:
print(df["Attrition"].head())

0    Yes
1     No
2    Yes
3     No
4     No
Name: Attrition, dtype: str


In [9]:
print(df["Attrition"].value_counts()) #target variable details

Attrition
No     1233
Yes     237
Name: count, dtype: int64


In [11]:
attrition_counts = df["Attrition"].value_counts()
attrition_rate = (attrition_counts['Yes']/row)*100
print(f"{attrition_rate:.2f}%")

16.12%


In [14]:
numeric_cols = df.select_dtypes(include=["number"]).columns
categorical_cols = df.select_dtypes(include=["object", "string"]).columns

In [15]:
print(f"Number of Numeric Columns: {len(numeric_cols)}")
print(f"Number of Categorical Columns: {len(categorical_cols)}")

Number of Numeric Columns: 26
Number of Categorical Columns: 9


Is the attrition rate balanced or imbalanced?
The dataset contains 1,470 employee records, of which 237 employees (16.12%) left the company and 1,233 employees (83.88%) stayed. This indicates that the dataset is imbalanced, as the majority of records belong to employees who did not leave. Therefore, relying solely on accuracy may be misleading, and evaluation metrics such as Precision, Recall, F1-score, and ROC-AUC should be used to assess model performance more effectively.

Task 2 - Data Cleaning & Preprocessing

In [17]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0] #no missing values acc to output 

Series([], dtype: int64)

In [18]:
columns_to_drop = [
    "EmployeeNumber",   
    "EmployeeCount",   
    "Over18",           
    "StandardHours"    
]

df.drop(columns=columns_to_drop, inplace=True) #irrelevant data dropped

In [19]:
binary_cols = ["Gender", "OverTime"]
binary_mapping = {"Yes": 1, "No": 0, "Male": 1, "Female": 0}

for col in binary_cols:
    df[col] = df[col].map(binary_mapping)

df["Attrition"] = df["Attrition"].map({
    "Yes": 1,
    "No": 0
}) #applied binary encoding to all binary categorical columns